In [27]:
import pandas as pd
import numpy as np
import nltk
import tensorflow as tf
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [35]:
import pandas as pd

# 1. Create a dictionary with real intents, user questions, and bot answers
real_data = {
    'User Utterance': [
        "Hello", "Hi there", "How are you?", "Good morning", "Hey",
        "I need help", "Can you assist me?", "I have a problem", "Help me please",
        "Where is my order?", "Track my package", "Order status", "Where is my box?",
        "Thank you", "Thanks a lot", "I appreciate it",
        "Goodbye", "See you later", "Bye", "Talk to you later"
    ],
    'Entities': [
        "greeting", "greeting", "greeting", "greeting", "greeting",
        "support", "support", "support", "support",
        "tracking", "tracking", "tracking", "tracking",
        "gratitude", "gratitude", "gratitude",
        "goodbye", "goodbye", "goodbye", "goodbye"
    ],
    'Bot Response': [
        "Hi there! How can I help you today?", "Hello! Hope you're having a good day.", "I'm doing great! How can I assist?", "Good morning! What's on your mind?", "Hey! What can I do for you?",
        "I'd be happy to help! What seems to be the issue?", "Of course, I am here to assist you.", "Please tell me more about the problem so I can help.", "I can definitely help with that!",
        "To track your order, please provide your 10-digit order ID.", "I can check your order status. What is your tracking number?", "Let's find your package! Please enter your order number.", "I can help you locate that. What is the order ID?",
        "You are very welcome!", "Happy to help!", "Anytime! Let me know if you need anything else.",
        "Goodbye! Have a great day!", "See you later! Feel free to reach out if you need anything else.", "Bye! Take care!", "Goodbye! Stay safe!"
    ]
}

# 2. Convert it to a DataFrame
new_df = pd.DataFrame(real_data)

# 3. Save it as your new chatbot_dataset.csv
new_df.to_csv('/content/chatbot_dataset.csv', index=False)

print("Success! Your brand new, real English dataset has been created.")

Success! Your brand new, real English dataset has been created.


In [36]:
# 1. Isolate the relevant columns and drop any missing values
# We will use 'User Utterance' as the input and 'Entities' as the target Intent
new_df = new_df[['User Utterance', 'Entities', 'Bot Response']].dropna()

# 2. Define a function to preprocess the text
def preprocess_text(text):
    # Tokenize and convert to lowercase
    tokens = nltk.word_tokenize(text.lower())

    # Lemmatize and keep only alphanumeric words (removes punctuation)
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in tokens if word.isalnum()]

    # Join back into a single string
    return " ".join(lemmatized_tokens)

# Apply the preprocessing function to the user utterances
new_df['Processed_Utterance'] = new_df['User Utterance'].apply(preprocess_text)

# 3. Encode the target labels (Entities) into integers
encoder = LabelEncoder()
new_df['Intent_Label'] = encoder.fit_transform(new_df['Entities'])

# Check the results of our preprocessing
new_df[['User Utterance', 'Processed_Utterance', 'Entities', 'Intent_Label']].head()

,User Utterance,Processed_Utterance,Entities,Intent_Label
0,Hello,hello,greeting,2
1,Hi there,hi there,greeting,2
2,How are you?,how are you,greeting,2
3,Good morning,good morning,greeting,2
4,Hey,hey,greeting,2


In [37]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical

# 1. Vectorize the text (Convert words to numbers using TF-IDF)
tokenizer = Tokenizer()
tokenizer.fit_on_texts(new_df['Processed_Utterance'])
X = tokenizer.texts_to_matrix(new_df['Processed_Utterance'], mode='tfidf')

# 2. Prepare the target labels (One-hot encoding for the neural network)
y = to_categorical(new_df['Intent_Label'])

# 3. Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Build the Neural Network
model = Sequential()
# Input layer
model.add(Dense(128, input_shape=(X_train.shape[1],), activation='relu'))
model.add(Dropout(0.5)) # Dropout helps prevent overfitting
# Hidden layer
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
# Output layer (Softmax gives us probabilities for each intent)
model.add(Dense(y.shape[1], activation='softmax'))

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# 5. Train the model
print("Training the chatbot model...")
history = model.fit(X_train, y_train, epochs=20, batch_size=8, validation_data=(X_test, y_test))

Training the chatbot model...
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 400ms/step - accuracy: 0.1875 - loss: 1.8943 - val_accuracy: 0.2500 - val_loss: 1.7241
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 302ms/step - accuracy: 0.1875 - loss: 1.6435 - val_accuracy: 0.2500 - val_loss: 1.7275
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.1875 - loss: 1.7883 - val_accuracy: 0.2500 - val_loss: 1.7349
Epoch 4/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.3750 - loss: 1.6692 - val_accuracy: 0.2500 - val_loss: 1.7398
Epoch 5/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.1250 - loss: 1.6946 - val_accuracy: 0.2500 - val_loss: 1.7426
Epoch 6/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.3750 - loss: 1.6275 - val_accuracy: 0.2500 - val_loss: 1.7500
Epoch 7/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.1875 - loss: 1.5760 - val_accuracy: 0.2500 - val_loss: 1.7550
Epoch 8/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.3125 - loss: 1.4909 - val_accuracy: 0.2500 - val_loss: 1.7610
Epoch 9/2

In [38]:
import random

def chat_with_bot(user_message):
    # 1. Preprocess the user's input exactly like we did the training data
    cleaned_text = preprocess_text(user_message)

    # 2. Convert the text into numbers (TF-IDF vector)
    input_vector = tokenizer.texts_to_matrix([cleaned_text], mode='tfidf')

    # 3. Ask the neural network to predict the intent
    predicted_probs = model.predict(input_vector, verbose=0)
    predicted_label_index = np.argmax(predicted_probs, axis=1)[0]

    # 4. Convert the predicted number back into the text label
    predicted_intent = encoder.inverse_transform([predicted_label_index])[0]

    # 5. Find all responses in the NEW dataset that match this intent
    possible_responses = new_df[new_df['Entities'] == predicted_intent]['Bot Response'].tolist()

    # 6. Pick a random response to send back
    if possible_responses:
        bot_reply = random.choice(possible_responses)
    else:
        bot_reply = "I'm sorry, I don't understand."

    print(f"User: {user_message}")
    print(f"Bot Intent Prediction: [{predicted_intent}]")
    print(f"Bot: {bot_reply}\n")

# Let's test it out with our new English phrases!
chat_with_bot("Hello there!")
chat_with_bot("Can you track my package?")

User: Hello there!
Bot Intent Prediction: [greeting]
Bot: Hi there! How can I help you today?

User: Can you track my package?
Bot Intent Prediction: [tracking]
Bot: I can check your order status. What is your tracking number?



In [39]:
import pickle

# 1. Save the trained neural network model
model.save("chatbot_model.keras")

# 2. Save the Tokenizer (so the app knows your vocabulary)
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

# 3. Save the Label Encoder (so the app knows the intent labels)
with open('encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)

print("Success! AI brain saved to disk.")

Success! AI brain saved to disk.


In [42]:
%%writefile app.py
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import streamlit as st
import pandas as pd
import numpy as np
import nltk
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.models import load_model
import pickle
import random

# --- PAGE CONFIGURATION ---
# This changes the browser tab title and expands the layout
st.set_page_config(page_title="AI Assistant", page_icon="✨", layout="centered")

# Load all the saved files
@st.cache_resource
def load_ai_components():
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
    nltk.download('wordnet', quiet=True)

    model = load_model("chatbot_model.keras")
    with open('tokenizer.pkl', 'rb') as f:
        tokenizer = pickle.load(f)
    with open('encoder.pkl', 'rb') as f:
        encoder = pickle.load(f)
    df = pd.read_csv('/content/chatbot_dataset.csv')
    return model, tokenizer, encoder, df

model, tokenizer, encoder, df = load_ai_components()
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = nltk.word_tokenize(text.lower())
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in tokens if word.isalnum()]
    return " ".join(lemmatized_tokens)

# --- CUSTOM CSS ---
# Adds a custom font color to the header and a light background
st.markdown("""
<style>
    .stApp {
        background-color: #f8f9fa;
    }
    h1 {
        color: #2c3e50;
        font-family: 'Helvetica Neue', sans-serif;
    }
</style>
""", unsafe_allow_html=True)

# --- SIDEBAR ---
with st.sidebar:
    st.title("🤖 About this Bot")
    st.write("This is a custom NLP chatbot built with **TensorFlow**, **NLTK**, and **Streamlit**.")
    st.divider()
    st.write("💡 **Try asking:**")
    st.write("- *Hello!*")
    st.write("- *Where is my order?*")
    st.write("- *I need help.*")

# --- MAIN CHAT UI ---
st.title("✨ My Smart NLP Assistant")
st.caption("🚀 Powered by Deep Learning")

# Initialize chat history with a default welcome message
if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "assistant", "content": "Hi there! I am your AI assistant. How can I help you today?"}
    ]

# Display chat messages with custom avatars!
for message in st.session_state.messages:
    avatar = "👤" if message["role"] == "user" else "🤖"
    with st.chat_message(message["role"], avatar=avatar):
        st.markdown(message["content"])

# Chat input box
if user_input := st.chat_input("Type your message here..."):
    # 1. Show user message with avatar
    st.chat_message("user", avatar="👤").markdown(user_input)
    st.session_state.messages.append({"role": "user", "content": user_input})

    # 2. AI Prediction
    cleaned_text = preprocess_text(user_input)
    input_vector = tokenizer.texts_to_matrix([cleaned_text], mode='tfidf')
    predicted_probs = model.predict(input_vector, verbose=0)
    predicted_label_index = np.argmax(predicted_probs, axis=1)[0]
    predicted_intent = encoder.inverse_transform([predicted_label_index])[0]

    # 3. Get Bot response
    possible_responses = df[df['Entities'] == predicted_intent]['Bot Response'].dropna().tolist()
    bot_reply = random.choice(possible_responses) if possible_responses else "I'm sorry, I don't understand."

    # 4. Show bot message with avatar
    with st.chat_message("assistant", avatar="🤖"):
        st.markdown(bot_reply)
    st.session_state.messages.append({"role": "assistant", "content": bot_reply})

Overwriting app.py


In [43]:
import time
from google.colab import output

!pkill -f streamlit
!streamlit run app.py --server.headless true --server.enableCORS false --server.enableXsrfProtection false &> logs.txt &

print("Booting up the AI server...")
time.sleep(4)
print("Your app should load below:")
output.serve_kernel_port_as_iframe(8501, height=700)

Booting up the AI server...
Your app should load below:


<IPython.core.display.Javascript object>